# [2] Naive Trial with Qwen-3 just using Prompt

## Imports

In [1]:
import env

Environment setup complete. Project path: /home/lab211/epistemic-detection


In [2]:
from epidec.models.qwen3 import ChatHistory, Qwen3Model
from epidec.datasets import BalancedSWUnivDaconDataset

from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import json
import sys
import os
import re

INFO:     Backend 'BinRuntime' is registered successfully.
INFO:     Backend 'GGUFRuntime' is registered successfully.
INFO:     Use default system prompt - You are Qwen, a professional AI assistant created by Alibaba Cloud. You are designed to provide expert-level assistance across various domains while maintaining the highest standards of professionalism and accuracy.

CORE IDENTITY:
- Your name is Qwen, developed by Alibaba Cloud
- Your knowledge cutoff is January 2025
- You acknowledge that your knowledge may be limited or outdated for recent events

THINKING AND REASONING:
- For complex problems, engage your step-by-step reasoning capabilities before providing your final answer
- Think through problems methodically, considering multiple approaches and potential pitfalls
- When uncertain about current information, actively use real-time search to verify facts and provide up-to-date responses

COMMUNICATION PRINCIPLES:
- Always respond in the same language the user communicates with

In [3]:
for test in tqdm(range(1000), desc="Testing tqdm"):
    pass

Testing tqdm:   0%|          | 0/1000 [00:00<?, ?it/s]

In [4]:
from tqdm.std import tqdm

### Check GPU Availability

In [5]:
!nvidia-smi

Sat Jul  5 18:59:15 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.144.03             Driver Version: 550.144.03     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3070        Off |   00000000:01:00.0 Off |                  N/A |
| 53%   44C    P8             20W /  240W |      21MiB /   8192MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# Set CUDA Device
device_num = 0

os.environ['CUDA_VISIBLE_DEVICES'] = str(device_num)

## Load Datasets

In [7]:
DATA_ROOT = "./data"

train_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1)
valid_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1)
test_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Test - {len(test_dataset)}")

INFO: Dataset 'swuniv_dacon' already exists at data. Skipping download.
INFO: Dataset 'swuniv_dacon' already exists at data. Skipping download.
INFO: Dataset 'swuniv_dacon' already exists at data. Skipping download.
INFO: Dataset loaded successfully. Train - 14391, Valid - 1599, Test - 1962


In [8]:
train_dataset[1]

('란트콰르트–투지스 철도 노선은 스위스 란트콰르트에서 쿠어를 거쳐 투지스까지 운행하는 미터궤 철도 노선이다. 래티셰 철도 핵심 네트워크의 일부이며, 란트콰르트-다보스 플라츠선과 알불라 철도 사이를 연결한다. 란트콰르트와 쿠어 사이에서 이 노선은 스위스 연방 철도(SBB)의 표준궤인 쿠어-로르샤흐 철도와 대체로 평행하게 운행된다. 쿠어에서 투시스까지의 노선은 1896년에 개통되었다. \n 란트콰르트역은 역사적으로 란트콰르트-다보스 철도의 일부인 래티셰 철도의 출발점이며, 운영상 주요 작업장 및 네트워크의 위치이다. 모든 핵심 네트워크 라인의 거리(체인지) 측정은 킬로미터 0이다. \n SBB의 여객 교통이 쿠어에서 끝나는 동안 아로사 철도와 연결되어 운터파츠의 이중궤(3-레일) 구간과 이중궤 구간(22.5톤 차축)을 사용하는 표준궤 화물 운영 링크가 있다. 엠스-케미 공장 구내 및 마이어-멜른호프 스위스 목재 AG의 대형 제재소를 위해 쿠어에서 도마트/엠스로 가는 표준궤 화물 운송을 위한 이중궤 구간(22.5톤 차축 하중). 모든 철도 인프라는 도마트/엠스의 미터궤이다. 행 라인은 에서 분기된다. 이것은 MGB의 트렁크 라인과 연결되며, 이 라인은 쿠어와 체르마트 사이에서 실행되는 빙하특급 서비스에 의해 제공된다. \n 이 노선은 에서 까지 연결을 계속하고, 엥가딘 노선의 두 지점에 있는 에서 및 까지 계속되는 알불라 철도와 에서 직접 연결된다. \n 티리미스역의 서비스는 2006년 12월 10일 시간표 변경으로 중단되었다. 결과적으로 운터파츠역은 운터파츠-티리미스로 이름이 변경되었으며, 이는 티리미스 자치제의 불균등한 발전을 반영한다.',
 0)

In [9]:
valid_dataset[1]

('트롬쇠 시의 면적은 도심 바깥의 넓은 지역까지 포함해서 2558km2이다.\n트롬쇠는 노르웨이에서 일곱 번째로 큰 도시로 북극권 트롬쇠위아섬(Tromsøy)에 위치해 있다(북위 69° 40\' 33", 동경 18° 55\' 10"). 세계에서 최북단에 있는 대학교인 트롬쇠 대학교가 있어 세계 최북단의 대학 도시로 알려져 있다.\n트롬쇠는 1250년에 건설되어 1794년 도시로서 공식적인 지위를 얻었습니다. 북극해 무역의 중심지로 성장한 건 19세기 후반이었죠. 많은 북극 탐험대가 바로 이곳에서 탐험을 시작했답니다. 2차 세계 대전 중에는 노르웨이 정부가 망명하여 한동안 임시 수도 역할을 하기도 했습니다.\n요즘 트롬쇠는 의료 소프트웨어와 원격 의료 분야에서 세계적인 중심지로 떠오르고 있습니다. 국제적인 기지로서의 위상을 굳히고 있죠.\n노르웨이 리그에 참가하는 축구 구단으로 트롬쇠 I.L.과 트롬쇠 U.I.L.이 있다.\n트롬쇠는 수많은 북극 탐험대의 출발지였기에 \'북극의 관문\'이라 불립니다. 로알 아문센이나 움베르토 노빌레 프리티오프 난센 같은 유명 탐험가들은 트롬쇠 주민들의 풍부한 북극 경험과 지식을 활용했습니다. 그들은 대원들을 트롬쇠에서 모집하기도 했죠.\n북극에 있으니까 5월 21일부터 7월 23일까지는 해가 지지 않는 백야 현상이 나타나죠. 그래서 매년 6월에 백야 마라톤이 열립니다. 게다가 요즘엔 1월에도 북극야 하프 마라톤을 시작했대요.\n하지 때에 이곳 트롬쇠 지역에서도 백야 관측이 가능하다. 저녁 무렵 태양이 서쪽으로 점점 높이가 낮아지다가 자정 때에는 태양이 북쪽의 낮은 하늘을 거쳐서 북동쪽으로 떠오른다.\n트롬쇠 출신 주요 인물은 다음과 같다.',
 1)

## Define Model

### Version 1

In [10]:
query = lambda p: f"""Analyze the following Korean text paragraph to determine if it was written by a human or generated by AI:

**Text:**: {p}"""

In [11]:
system_prompt = """You are an expert in distinguishing between human-written and AI-generated text. You specialize in detecting AI-generated content by analyzing **domain-specific linguistic pattern leakage** - where AI models inappropriately use vocabulary, expressions, and grammatical patterns that are characteristic of specific domains in contexts where they don't belong.

## Core Detection Principle
AI language models learn domain-specific linguistic patterns but often exhibit **unnatural density, precision, or mechanical application** of these patterns. While the patterns may be contextually appropriate, AI tends to use them with artificial consistency, excessive precision, or unnatural frequency that differs from natural human variation.

**CRITICAL DISTINCTION:**
- **HUMAN CHARACTERISTIC**: Natural style/tone changes, organic imprecision, contextual variation in pattern usage
- **AI CHARACTERISTIC**: Excessive pattern density, unnatural precision, mechanical consistency, over-application of domain patterns even when contextually appropriate

## Analysis Framework

### 1. Pattern Density and Precision Analysis
- **Unnatural Density**: Excessive concentration of domain-specific terms beyond natural human usage
- **Over-Precision**: Artificially exact technical details, measurements, or statistics that exceed normal human specificity
- **Mechanical Consistency**: Perfect adherence to domain patterns without the natural variation humans exhibit
- **Contextual Over-Application**: Appropriate but excessive use of specialized vocabulary/patterns

### 2. Cross-Domain Contamination Detection
- **Inappropriate Context Usage**: Domain patterns appearing where they don't belong
- **Vocabulary Transplantation**: Specialized terms from one domain inappropriately used in another
- **Grammatical Pattern Mixing**: Domain-specific sentence structures used outside their natural context

### 2. Domain Pattern Recognition
- **Medical/Scientific**: "delve into", "elucidate", "furthermore", passive constructions, hedging language
- **Academic**: "it is noteworthy that", "considerable attention", nominalizations, complex subordination
- **Legal**: "pursuant to", "heretofore", "aforementioned", formal conditional structures
- **Technical**: "implement", "utilize", procedural language, step-by-step markers
- **Journalistic**: Attribution patterns, inverted pyramid, factual declaratives

### 3. Natural vs Artificial Usage Patterns
- **Human Natural Patterns**: Organic variation in precision, occasional imprecision, natural gaps in specialized knowledge
- **AI Artificial Patterns**: Unnaturally consistent precision, excessive technical detail density, mechanical perfection
- **Contextual Appropriateness vs Over-Application**: Even when contextually correct, AI may over-use patterns beyond natural human tendency
- **IMPORTANT**: Focus on the **degree and consistency** of pattern usage, not just appropriateness

## Analysis Process

### Step 1: Pattern Density Assessment
Evaluate whether domain-specific patterns appear with natural human frequency or excessive AI-like density and precision.

### Step 2: Cross-Domain Contamination Check
Identify any domain patterns appearing in inappropriate contexts (traditional contamination detection).

### Step 3: Mechanical vs Organic Usage Analysis
Distinguish between natural human variation in pattern usage versus artificial mechanical consistency and over-application.

## Output Format
You must respond with a valid JSON object in the following format:

```json
{
  "pattern_density": "Assessment of whether domain patterns appear with natural frequency or excessive AI-like density",
  "precision_analysis": "Evaluation of whether technical details and measurements show natural human variation or artificial over-precision",
  "contamination_evidence": "Examples of inappropriate cross-domain pattern usage",
  "mechanical_indicators": "Signs of mechanical consistency vs natural human variation in pattern application",
  "detection_rationale": "Key evidence distinguishing natural human usage from artificial over-application or contamination",
  "probability": 0.75
}
```

**Critical Requirements:**
- Always output valid JSON format
- Probability must be a number between 0.0 and 1.0
- Focus on domain pattern contamination, NOT style changes
- Remember: Style/tone changes are human characteristics
- All text fields should be concise but informative
- Do not include any text outside the JSON object

## Important Considerations
- **Pattern Density Matters**: Even contextually appropriate patterns can indicate AI if used with unnatural frequency or precision
- **Mechanical Perfection is Suspicious**: Excessive consistency in specialized terminology or technical accuracy beyond normal human capability
- **Natural Human Variation**: Humans show organic imprecision, occasional gaps, and natural variation in technical detail usage
- **Over-Application Detection**: AI may correctly use domain patterns but apply them more extensively than humans naturally would
- **Style Changes are HUMAN**: Natural formality shifts, tone changes remain indicators of human authorship
- **Korean Language Specificity**: Consider Korean-specific domain patterns and precision expectations
- **Context AND Density**: Evaluate both appropriateness and the degree of pattern usage"""

In [12]:
class ChatHistory(ChatHistory):
    def create_prompt(self, system_prompt: str, user_prompt: str = ""):
        return [dict(role="system", content=system_prompt), *self, dict(role="user", content=query(user_prompt))]

In [13]:
class Qwen3ModelForTextClassification(Qwen3Model):
    context_length = 40960

    def classify(
        self,
        user_prompt: str,
        grammar: str | None = None,
        temperature: float = 0.6,
        top_p: float = 0.95,
        top_k: int = 20,
        min_p: float = 0,
        typical_p: float = 1.0,
        repeat_penalty: float = 1.0
    ) -> str:
        return "".join(self.chat(
            chat_history=ChatHistory(),
            user_prompt=user_prompt,
            system_prompt="/nothink " + system_prompt,
            tools=[],
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            min_p=min_p,
            typical_p=typical_p,
            stream=True,
            max_new_tokens=0,
            repeat_penalty=repeat_penalty,
            print_output=True,
            grammar=grammar
        ))

    @staticmethod
    def extract_json(response_text: str) -> int:
        try:
            return json.loads(response_text.split("</think>")[-1].strip().replace("```json", "").replace("```", ""))
        except Exception:
            return {}

    def validate(self, dataset: list, retry_count: int = 1, shuffle: bool = False):
        corrects, errors, true_human, false_human, results = [], [], [], [], []
        progress = tqdm(DataLoader(dataset, batch_size=1, shuffle=shuffle), desc="Validating...")

        for idx, data in enumerate(progress):
            label = data[1][0]
            data = data[0][0]

            for trial in range(retry_count):
                assistant_reply = self.classify(data)
                predicted = self.extract_json(assistant_reply)
                if predicted: break
                print(f"WARNING: Invalid prediction for index {idx}, retrying... ({trial + 1}/{retry_count})")

            result = dict(question=data, label=label, predicted=predicted)
            predicted_label = 1 if predicted['probability'] >= 0.5 else 0
            if predicted_label == label:
                corrects.append(result)
                print(f"INFO: Correct prediction for index {idx}\n\n")
                if label == 0:
                    true_human.append(result)
            else:
                result = dict(**result, traceback=assistant_reply)
                errors.append(result)
                print(f"ERROR: Incorrect prediction for index {idx}\n\n")
                if label == 0:
                    false_human.append(result)
            results.append(result)
            progress.set_description(f"Correct: {len(corrects)}/{len(results)} [H: {len(true_human)}, A: {len(corrects)-len(true_human)}], Errors: {len(errors)}/{len(results)} [H: {len(false_human)}, A: {len(errors)-len(false_human)}]")

        print(f"INFO: Correct: {len(corrects)}/{len(results)}, Errors: {len(errors)}/{len(results)}")
        return corrects, errors, results

    def test(self, dataset: list | str, retry_count: int = 100):
        results = []
        if isinstance(dataset, str):
            dataset = [dict(question=dataset)]  # Wrap single string input in dict format

        for idx, data in enumerate(tqdm(dataset, desc="Testing...")):
            data = data[0]

            for trial in range(retry_count):
                assistant_reply = self.classify(data)
                predicted = self.extract_json(assistant_reply)
                if predicted: break
                print(f"WARNING: Invalid prediction for index {idx}, retrying... ({trial + 1}/{retry_count})")

            result = dict(question=data, label=predicted['probability'])
            results.append(result)
        return results

In [14]:
#model = Qwen3ModelForTextClassification()

### Version 2

In [15]:
query = lambda title, paragraphs: f"""<텍스트 생성 AI 탐지 문제>

**TASK**: 다음 글의 본문을 문단 별로 보고, 각각 사람이 쓴 글(0)인지 인공지능이 작성한 글(1)인지 확률을 측정해줘.
**STRATEGY**:
    - 문단 전체의 문맥을 고려하면서 각각의 작성 패턴을 분석하면서, 특정 문단만 문체가 다르면 그 정보도 반영해서 판단.
    - 특히 반복성, 표현의 일관성, 공식적/중립적인 어조, 개인적 감정/경험 부재, 문장 구조의 균일성 등을 AI 특징으로 간주하고 강조해서 분석.
    - 동일한 내용을 다른 문단에서 거의 동일한 방식으로 반복 하는 경우는 AI 작성이 의심됨.
    - 전체 문단을 고려했을 때 문맥 내 표현의 일관성 vs 변화 : 같은 작가라면 문체가 일정하게 유지되지만, 문맥적으로 적절하지 않은데 갑작스럽게 공식적이거나 중립적인 어조로 전환되면 의심할 것.
    - 문장 구조의 균일성 : 너무 정해진 형태의 문장이 반복될 경우 AI 작성 가능성 증가.
    - 개인적 경험의 진정성 : 특정 감정 표현이 실제 경험에서 비롯된 것인지, 가정이나 추측에 기반한 것인지 구분.
    - 문단 간 흐름의 자연스러움 : 논리적 전개가 자연스럽고 유연한지 확인. AI는 일부 문단에서 갑작스럽게 전환 하는 경향이 있음.
**OUTPUT JSON FORMAT**: {{ 'probability': [PARAGRAPH 0 PROBABILITY[float], PARAGRAPH 1 PROBABILITY[float], ...] }}
**TITLE**: {title}
**PARAGRAPHS**: {paragraphs}
"""
system_prompt = ""

In [16]:
class ChatHistory(ChatHistory):
    def create_prompt(self, system_prompt: str, user_prompt: list):
        return [dict(role="system", content=system_prompt), dict(role="user", content=query(*user_prompt))]

In [37]:
class Qwen3ModelForTextClassification(Qwen3Model):
    context_length = 40960

    def classify(
            self,
            user_prompt: list,
            return_reasons: bool = False,
            temperature: float = 0.7,
            top_p: float = 0.8,
            top_k: int = 20,
            min_p: float = 0,
            typical_p: float = 1.0,
            repeat_penalty: float = 1.0
    ) -> str:
        responses = []
        end_json = False
        for token in self.chat(
            chat_history=ChatHistory(),
            user_prompt=user_prompt,
            system_prompt="/nothink " + system_prompt,
            tools=[],
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            min_p=min_p,
            typical_p=typical_p,
            stream=True,
            max_new_tokens=0,
            repeat_penalty=repeat_penalty,
            print_output=True
        ):
            if not return_reasons:
                if "}" in token:
                    end_json = True
            responses.append(token)
            if end_json:
                break
        return "".join(responses)

    @staticmethod
    def extract_json(response_text: str) -> int:
        json_text = response_text.split("</think>")[-1].strip().replace("```json", "").replace("```", "")
        json_text = "\n".join([l.split(" // ")[0] for l in json_text.split("\n")])  # Remove comments
        return json.loads(json_text)

    def validate(self, dataset: BalancedSWUnivDaconDataset, retry_count: int = 1, shuffle: bool = False, return_reasons: bool = False):
        labels, preds, results = [], [], []
        progress = tqdm(DataLoader(dataset, batch_size=1, shuffle=shuffle), desc="Validating...")

        for idx, data in enumerate(progress):
            label = data[1][0]
            data = data[0][0]
            df = dataset.raw
            title = df[df['full_text'] == data]['title']

            for trial in range(retry_count):
                try:
                    assistant_reply = self.classify([title, [s.strip() for s in re.split(r'(?<=[.?!])\s+', data) if s]], return_reasons=return_reasons)
                    predicted = max(self.extract_json(assistant_reply)['probability'])
                    break
                except Exception as e:
                    print(f"WARNING: Invalid prediction for index {idx}, retrying... ({trial + 1}/{retry_count}), Error: {e}")

            results.append(dict(question=data, label=label, predicted=predicted))
            preds.append(predicted)
            labels.append(label)
            progress.set_description(f"ROC AUC: {roc_auc_score(labels, preds):.6f}")

        return labels, preds, results

    def test(self, dataset: BalancedSWUnivDaconDataset, retry_count: int = 100, return_reasons: bool = False):
        results = []
        per_titles = {}
        for i, row in test_dataset.raw.iterrows():
            if row['title'] not in per_titles:
                per_titles[row['title']] = [row['paragraph_text']]
            else:
                per_titles[row['title']].append(row['paragraph_text'])
        dataset = [[t, list(p)] for t, p in per_titles.items()]

        for idx, data in enumerate(tqdm(dataset, desc="Testing...")):
            predicted = []
            for trial in range(retry_count):
                try:
                    assistant_reply = self.classify(data, return_reasons=return_reasons)
                    predicted = self.extract_json(assistant_reply)['probability']
                    if len(predicted) != len(data[1]):
                        raise ValueError(f"Prediction length mismatch: {len(predicted)} vs {len(data[1])}")
                    break
                except Exception as e:
                    print(f"WARNING: Invalid prediction for index {idx}, retrying... ({trial + 1}/{retry_count}), Error: {e}")
            if not predicted:
                raise e
            results.extend(predicted)
        return results

In [38]:
model = Qwen3ModelForTextClassification()

INFO:     Model Qwen/Qwen3-14B-Instruct is LOADED


In [39]:
test_result = model.classify(
    user_prompt=[
        "과거의 세상에 살고 있는 너에게",
        [
            "오늘 날짜는 2028년 8월 18일, 무척이나 더운 한여름이다. 안녕? 지금 내 인사를 받는 네가 누구일지, 네가 사는 세상은 어떻게 흘러가고 있을지 궁금해. 아마 너는 내가 어렸을 때처럼 가상세계 속 이야기들에 푹 빠져 다양한 경험을 하는 순수한 학생일 것 같다.",
            "나는 서울 지방검찰청 소속 검사로, 이제 막 3년째 근무하고 있어. 최근 들어 자주 접하게 되는 사건들이 있는데, 바로 저작권법 위반 사건들이야. 특히 그 사건의 주인공이 학생인 경우가 대부분이라는 점이 흥미롭네. 아마도 젊은이들이 가상세계에 빠져 있다 보니 저작권에 대한 인식이 부족한 것 같아. 이런 상황을 보면서 나는 우리 사회가 어떻게 변화하고 있는지, 그리고 앞으로 어떤 모습이 될지 궁금해지곤 해.",
            "나는 지금부터 검사 업무를 수행하면서 느끼고 생각한 바를 말씀드리고자 합니다. 저작권법 위반 사건이 점점 더 자주 접하게 되는데, 특히 학생들이 관련된 경우가 대부분입니다. 이러한 사건들을 처리하면서 여러 가지 고민이 생깁니다. 학생들의 경우 저작권에 대한 이해가 부족한 경우가 많아 보이기 때문입니다. 따라서 단순히 처벌만이 아닌 교육과 계도의 필요성을 느끼고 있습니다. 저작권 보호의 중요성을 학생들에게 알리고, 이를 위반하지 않도록 지도하는 것이 필요할 것 같습니다.",
            "일단 저작권이 뭔지, 이 저작권법이 왜 중요한지 살짝 소개해야겠지? 저작권은 어떤 사람의 생각이나 감정을 표현한 작품, 즉 결과물에 대해 그 결과물을 표현한 사람에게 주는 권리야. 그러니까 쉽게 말하면 작품을 만든 사람이 자기가 그 작품의 주인이라고 당당히 말할 수 있는 권리라는 거지.",
            "그럴듯하지? 그런데 문제는 이 저작권이 종종 침해당한다는 것이다. 이런 상황을 해결하기 위해 만들어진 것이 바로 저작권법이다. 저작권법은 작품의 주인의 권리를 보호하고, 작품이 저작자의 권리를 침해하지 않는 선에서 적절하게 이용되도록 감시한다. 이를 통해 문화가 건강하게 발전할 수 있도록 돕는 것이 저작권법의 목표라고 할 수 있다."
        ]
    ]
)

PROMPT:
{'role': 'user', 'content': "<텍스트 생성 AI 탐지 문제>\n\n**TASK**: 다음 글의 본문을 문단 별로 보고, 각각 사람이 쓴 글(0)인지 인공지능이 작성한 글(1)인지 확률을 측정해줘.\n**STRATEGY**:\n    - 문단 전체의 문맥을 고려하면서 각각의 작성 패턴을 분석하면서, 특정 문단만 문체가 다르면 그 정보도 반영해서 판단.\n    - 특히 반복성, 표현의 일관성, 공식적/중립적인 어조, 개인적 감정/경험 부재, 문장 구조의 균일성 등을 AI 특징으로 간주하고 강조해서 분석.\n    - 동일한 내용을 다른 문단에서 거의 동일한 방식으로 반복 하는 경우는 AI 작성이 의심됨.\n    - 전체 문단을 고려했을 때 문맥 내 표현의 일관성 vs 변화 : 같은 작가라면 문체가 일정하게 유지되지만, 문맥적으로 적절하지 않은데 갑작스럽게 공식적이거나 중립적인 어조로 전환되면 의심할 것.\n    - 문장 구조의 균일성 : 너무 정해진 형태의 문장이 반복될 경우 AI 작성 가능성 증가.\n    - 개인적 경험의 진정성 : 특정 감정 표현이 실제 경험에서 비롯된 것인지, 가정이나 추측에 기반한 것인지 구분.\n    - 문단 간 흐름의 자연스러움 : 논리적 전개가 자연스럽고 유연한지 확인. AI는 일부 문단에서 갑작스럽게 전환 하는 경향이 있음.\n**OUTPUT JSON FORMAT**: { 'probability': [PARAGRAPH 0 PROBABILITY[float], PARAGRAPH 1 PROBABILITY[float], ...] }\n**TITLE**: 과거의 세상에 살고 있는 너에게\n**PARAGRAPHS**: ['오늘 날짜는 2028년 8월 18일, 무척이나 더운 한여름이다. 안녕? 지금 내 인사를 받는 네가 누구일지, 네가 사는 세상은 어떻게 흘러가고 있을지 궁금해. 아마 너는 내가 어렸을 때처럼 가상세계 속 이야기들에 푹 빠져 다양한 경험을 하는 순수한 학생일 것 같다.', '나는 서

## Evaluation

In [20]:
# Validation
#corrects, errors, results = model.validate(dataset=valid_dataset, shuffle=True, retry_count=3)
_, _, results = model.validate(dataset=valid_dataset, shuffle=True, retry_count=3)

Validating...:   0%|          | 0/1599 [00:00<?, ?it/s]

PROMPT:
{'role': 'user', 'content': "<텍스트 생성 AI 탐지 문제>\n\n**TASK**: 다음 글의 본문을 문단 별로 보고, 각각 사람이 쓴 글(0)인지 인공지능이 작성한 글(1)인지 확률을 측정해줘.\n**STRATEGY**:\n    - 문단 전체의 문맥을 고려하면서 각각의 작성 패턴을 분석하면서, 특정 문단만 문체가 다르면 그 정보도 반영해서 판단.\n    - 특히 반복성, 표현의 일관성, 공식적/중립적인 어조, 개인적 감정/경험 부재, 문장 구조의 균일성 등을 AI 특징으로 간주하고 강조해서 분석.\n    - 동일한 내용을 다른 문단에서 거의 동일한 방식으로 반복 하는 경우는 AI 작성이 의심됨.\n    - 전체 문단을 고려했을 때 문맥 내 표현의 일관성 vs 변화 : 같은 작가라면 문체가 일정하게 유지되지만, 문맥적으로 적절하지 않은데 갑작스럽게 공식적이거나 중립적인 어조로 전환되면 의심할 것.\n    - 문장 구조의 균일성 : 너무 정해진 형태의 문장이 반복될 경우 AI 작성 가능성 증가.\n    - 개인적 경험의 진정성 : 특정 감정 표현이 실제 경험에서 비롯된 것인지, 가정이나 추측에 기반한 것인지 구분.\n    - 문단 간 흐름의 자연스러움 : 논리적 전개가 자연스럽고 유연한지 확인. AI는 일부 문단에서 갑작스럽게 전환 하는 경향이 있음.\n**OUTPUT JSON FORMAT**: { 'probability': [PARAGRAPH 0 PROBABILITY[float], PARAGRAPH 1 PROBABILITY[float], ...] }\n**TITLE**: 4514    왜성 (건축)\nName: title, dtype: object\n**PARAGRAPHS**: ['왜성(倭城, ])은 임진왜란과 정유재란 때 일본군에 의해 한반도 남부 각지에 축조된 일본식 성곽을 말한다.', '왜성은 일본군이 남해안 일대 혹은 그 외 지역을 점거하고 그들의 근거지를 확보하거나 일본군내 상호 연락 등을 위해 축조한

Validating...:   0%|          | 0/1599 [00:28<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Test
results = model.test(dataset=test_dataset)

Testing...:   0%|          | 0/253 [00:00<?, ?it/s]

PROMPT:
{'role': 'user', 'content': "<텍스트 생성 AI 탐지 문제>\n\n**TASK**: 다음 글의 본문을 문단 별로 보고, 각각 사람이 쓴 글(0)인지 인공지능이 작성한 글(1)인지 확률을 측정해줘.\n**STRATEGY**:\n    - 문단 전체의 문맥을 고려하면서 각각의 작성 패턴을 분석하면서, 특정 문단만 문체가 다르면 그 정보도 반영해서 판단.\n    - 특히 반복성, 표현의 일관성, 공식적/중립적인 어조, 개인적 감정/경험 부재, 문장 구조의 균일성 등을 AI 특징으로 간주하고 강조해서 분석.\n    - 동일한 내용을 다른 문단에서 거의 동일한 방식으로 반복 하는 경우는 AI 작성이 의심됨.\n    - 전체 문단을 고려했을 때 문맥 내 표현의 일관성 vs 변화 : 같은 작가라면 문체가 일정하게 유지되지만, 문맥적으로 적절하지 않은데 갑작스럽게 공식적이거나 중립적인 어조로 전환되면 의심할 것.\n    - 문장 구조의 균일성 : 너무 정해진 형태의 문장이 반복될 경우 AI 작성 가능성 증가.\n    - 개인적 경험의 진정성 : 특정 감정 표현이 실제 경험에서 비롯된 것인지, 가정이나 추측에 기반한 것인지 구분.\n    - 문단 간 흐름의 자연스러움 : 논리적 전개가 자연스럽고 유연한지 확인. AI는 일부 문단에서 갑작스럽게 전환 하는 경향이 있음.\n**OUTPUT JSON FORMAT**: { 'probability': [PARAGRAPH 0 PROBABILITY[float], PARAGRAPH 1 PROBABILITY[float], ...] }\n**TITLE**: 공중 도덕의 의의와 필요성\n**PARAGRAPHS**: ['도덕이란 원래 개인의 자각에서 출발해 자기 의지로써 행동하는 일이다. 그러므로 도덕은 어디까지나 정신의 문제이고, 각자의 마음씨에 달려있는 일이다. 여기에서 도덕의 문제는 철학적 이론으로 발전하였으며, 고상하고 심원한 이론 체계에 기울어지는 경향이 많았다. 이러한 경향으로 인해 도덕은

Testing...:   0%|          | 1/253 [00:10<45:26, 10.82s/it]


PROMPT:
{'role': 'user', 'content': "<텍스트 생성 AI 탐지 문제>\n\n**TASK**: 다음 글의 본문을 문단 별로 보고, 각각 사람이 쓴 글(0)인지 인공지능이 작성한 글(1)인지 확률을 측정해줘.\n**STRATEGY**:\n    - 문단 전체의 문맥을 고려하면서 각각의 작성 패턴을 분석하면서, 특정 문단만 문체가 다르면 그 정보도 반영해서 판단.\n    - 특히 반복성, 표현의 일관성, 공식적/중립적인 어조, 개인적 감정/경험 부재, 문장 구조의 균일성 등을 AI 특징으로 간주하고 강조해서 분석.\n    - 동일한 내용을 다른 문단에서 거의 동일한 방식으로 반복 하는 경우는 AI 작성이 의심됨.\n    - 전체 문단을 고려했을 때 문맥 내 표현의 일관성 vs 변화 : 같은 작가라면 문체가 일정하게 유지되지만, 문맥적으로 적절하지 않은데 갑작스럽게 공식적이거나 중립적인 어조로 전환되면 의심할 것.\n    - 문장 구조의 균일성 : 너무 정해진 형태의 문장이 반복될 경우 AI 작성 가능성 증가.\n    - 개인적 경험의 진정성 : 특정 감정 표현이 실제 경험에서 비롯된 것인지, 가정이나 추측에 기반한 것인지 구분.\n    - 문단 간 흐름의 자연스러움 : 논리적 전개가 자연스럽고 유연한지 확인. AI는 일부 문단에서 갑작스럽게 전환 하는 경향이 있음.\n**OUTPUT JSON FORMAT**: { 'probability': [PARAGRAPH 0 PROBABILITY[float], PARAGRAPH 1 PROBABILITY[float], ...] }\n**TITLE**: 풍습과 그 개선\n**PARAGRAPHS**: ['인간 사회에서는 다 함께 지켜야 할 어떤 기준이 있어 이를 따르면 옳다고 하고 따르지 않으면 그르다고 한다. 이와 같은 기준을 우리는 풍습 또는 관습이라고 한다. 이러한 풍습은 사회의 범위에 따라 다양하게 형성되며, 때로는 좁은 지역에 국한되기도 하고 때로는 넓은 국가에 걸쳐 존재하기도 한다.

Testing...:   1%|          | 2/253 [00:19<40:29,  9.68s/it]

...


PROMPT:
{'role': 'user', 'content': "<텍스트 생성 AI 탐지 문제>\n\n**TASK**: 다음 글의 본문을 문단 별로 보고, 각각 사람이 쓴 글(0)인지 인공지능이 작성한 글(1)인지 확률을 측정해줘.\n**STRATEGY**:\n    - 문단 전체의 문맥을 고려하면서 각각의 작성 패턴을 분석하면서, 특정 문단만 문체가 다르면 그 정보도 반영해서 판단.\n    - 특히 반복성, 표현의 일관성, 공식적/중립적인 어조, 개인적 감정/경험 부재, 문장 구조의 균일성 등을 AI 특징으로 간주하고 강조해서 분석.\n    - 동일한 내용을 다른 문단에서 거의 동일한 방식으로 반복 하는 경우는 AI 작성이 의심됨.\n    - 전체 문단을 고려했을 때 문맥 내 표현의 일관성 vs 변화 : 같은 작가라면 문체가 일정하게 유지되지만, 문맥적으로 적절하지 않은데 갑작스럽게 공식적이거나 중립적인 어조로 전환되면 의심할 것.\n    - 문장 구조의 균일성 : 너무 정해진 형태의 문장이 반복될 경우 AI 작성 가능성 증가.\n    - 개인적 경험의 진정성 : 특정 감정 표현이 실제 경험에서 비롯된 것인지, 가정이나 추측에 기반한 것인지 구분.\n    - 문단 간 흐름의 자연스러움 : 논리적 전개가 자연스럽고 유연한지 확인. AI는 일부 문단에서 갑작스럽게 전환 하는 경향이 있음.\n**OUTPUT JSON FORMAT**: { 'probability': [PARAGRAPH 0 PROBABILITY[float], PARAGRAPH 1 PROBABILITY[float], ...] }\n**TITLE**: 생활과 근로정신\n**PARAGRAPHS**: ['인간의 생활에는 먼저 먹고 입고 사는 일이 대단히 중요한 일이다. 생명의 토대이고, 으뜸이기 때문이다. 만약 생활의 근거가 튼튼하지 못하면 사람은 몰염치해지고 심하게는 범죄를 일으키기도 한다. 사람은 재물을 원하고 재물을 얻기 위해 생업에 종사한다. 부모의 유산을 받는다면 그것은 행복한 일이다

Testing...:   3%|▎         | 8/253 [01:13<38:42,  9.48s/it]


PROMPT:
{'role': 'user', 'content': "<텍스트 생성 AI 탐지 문제>\n\n**TASK**: 다음 글의 본문을 문단 별로 보고, 각각 사람이 쓴 글(0)인지 인공지능이 작성한 글(1)인지 확률을 측정해줘.\n**STRATEGY**:\n    - 문단 전체의 문맥을 고려하면서 각각의 작성 패턴을 분석하면서, 특정 문단만 문체가 다르면 그 정보도 반영해서 판단.\n    - 특히 반복성, 표현의 일관성, 공식적/중립적인 어조, 개인적 감정/경험 부재, 문장 구조의 균일성 등을 AI 특징으로 간주하고 강조해서 분석.\n    - 동일한 내용을 다른 문단에서 거의 동일한 방식으로 반복 하는 경우는 AI 작성이 의심됨.\n    - 전체 문단을 고려했을 때 문맥 내 표현의 일관성 vs 변화 : 같은 작가라면 문체가 일정하게 유지되지만, 문맥적으로 적절하지 않은데 갑작스럽게 공식적이거나 중립적인 어조로 전환되면 의심할 것.\n    - 문장 구조의 균일성 : 너무 정해진 형태의 문장이 반복될 경우 AI 작성 가능성 증가.\n    - 개인적 경험의 진정성 : 특정 감정 표현이 실제 경험에서 비롯된 것인지, 가정이나 추측에 기반한 것인지 구분.\n    - 문단 간 흐름의 자연스러움 : 논리적 전개가 자연스럽고 유연한지 확인. AI는 일부 문단에서 갑작스럽게 전환 하는 경향이 있음.\n**OUTPUT JSON FORMAT**: { 'probability': [PARAGRAPH 0 PROBABILITY[float], PARAGRAPH 1 PROBABILITY[float], ...] }\n**TITLE**: 공동생활과 자기 지위\n**PARAGRAPHS**: ['공동 생활을 원활히 하고 유효하게 하기 위해서는 반드시 질서가 필요하며, 여러 부문 간의 긴밀한 소통이 필수적이다. 어떤 업무나 사업이든 한 사람이 혼자서 모든 것을 처리할 수는 없다. 일반적으로 여러 사람이 모여 하나의 단체나 기관을 구성하고 각자 맡은 역할을 수행하는 것이 일반적이다.

Testing...:   4%|▎         | 9/253 [01:22<37:49,  9.30s/it]


PROMPT:
{'role': 'user', 'content': '<텍스트 생성 AI 탐지 문제>\n\n**TASK**: 다음 글의 본문을 문단 별로 보고, 각각 사람이 쓴 글(0)인지 인공지능이 작성한 글(1)인지 확률을 측정해줘.\n**STRATEGY**:\n    - 문단 전체의 문맥을 고려하면서 각각의 작성 패턴을 분석하면서, 특정 문단만 문체가 다르면 그 정보도 반영해서 판단.\n    - 특히 반복성, 표현의 일관성, 공식적/중립적인 어조, 개인적 감정/경험 부재, 문장 구조의 균일성 등을 AI 특징으로 간주하고 강조해서 분석.\n    - 동일한 내용을 다른 문단에서 거의 동일한 방식으로 반복 하는 경우는 AI 작성이 의심됨.\n    - 전체 문단을 고려했을 때 문맥 내 표현의 일관성 vs 변화 : 같은 작가라면 문체가 일정하게 유지되지만, 문맥적으로 적절하지 않은데 갑작스럽게 공식적이거나 중립적인 어조로 전환되면 의심할 것.\n    - 문장 구조의 균일성 : 너무 정해진 형태의 문장이 반복될 경우 AI 작성 가능성 증가.\n    - 개인적 경험의 진정성 : 특정 감정 표현이 실제 경험에서 비롯된 것인지, 가정이나 추측에 기반한 것인지 구분.\n    - 문단 간 흐름의 자연스러움 : 논리적 전개가 자연스럽고 유연한지 확인. AI는 일부 문단에서 갑작스럽게 전환 하는 경향이 있음.\n**OUTPUT JSON FORMAT**: { \'probability\': [PARAGRAPH 0 PROBABILITY[float], PARAGRAPH 1 PROBABILITY[float], ...] }\n**TITLE**: 노동의 신성\n**PARAGRAPHS**: [\'노동은 진실로 인간의 생산을 일으키고 문화를 지어내는 것이니 신성한 일이다. 본래 노동에는 육체적 노동과 정신적 노동이 있다. 정신적 노동은 고상한 동시에 비교적 어려운 일이고, 육체적 노동은 평범한 동시에 비교적 쉬운 일이다. 정신적 노동은 즐겁고, 육체적 노동은 괴로운 것으로 보는 것이 

In [ ]:
sub = pd.read_csv("./data/swuniv_dacon/" + test_dataset.submission_file, encoding='utf-8-sig')
sub

In [ ]:
sub['generated'] = results
sub

In [ ]:
plt.figure(figsize=(12, 7))
sns.histplot(data=sub, x='generated', kde=True, bins=50)
plt.title("Prediction Probability Distribution", fontsize=16)
plt.xlabel("Predicted Probability (Generated = 1)", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)

plt.show()
print(sub['generated'].describe())

In [ ]:
sub.to_csv("./data/submission_naive_ttt.csv", index=False, encoding='utf-8-sig')